# DANDI and NWB: Remote Streaming

Browse an openly shared DANDI dandiset, explicitly choose a discovered NWB asset, and stream it without downloading the complete file.

## Quick start

Run this notebook from top to bottom. The default result is a local synthetic NWB file created in a temporary directory, inspected, and removed automatically. Remote DANDI streaming is an optional later section; leave its identifiers and selection index unset to stay offline.

## Prerequisites

- Python 3.9+ and familiarity with HDF5 groups and datasets.
- A public dandiset selected from https://dandiarchive.org/ only for the optional remote portion.
- HTTP range streaming avoids full-file downloads, but large requested slices can still transfer substantial data.

## Setup

Install the minimal dependencies for the default local synthetic NWB tutorial. The DANDI client and `remfile` are installed separately only for optional remote streaming.

In [ ]:
# Default local synthetic-NWB dependencies
!pip install -q h5py pynwb numpy

In [ ]:
# Optional remote-streaming dependencies; run only after choosing a public dandiset.
# !pip install -q dandi remfile

## Discover and explicitly select an NWB asset

After reviewing a public dandiset's landing page, enter its identifier and version. This cell lists the actual `.nwb` paths. Set `SELECTED_ASSET_INDEX` to one printed index and rerun. Selection remains disabled by default and the index is validated against freshly discovered paths, preventing accidental use of a first or stale asset.

In [ ]:
DANDISET_ID = None  # Set only after reviewing a public DANDI landing page.
DANDISET_VERSION = 'draft'  # Prefer a published version string for reproducibility.
SELECTED_ASSET_INDEX = None  # Set to one printed index after reviewing the list, then rerun.
MAX_ASSETS_TO_SHOW = 30

selected_asset_path = None
if DANDISET_ID:
    try:
        from dandi.dandiapi import DandiAPIClient
        with DandiAPIClient() as client:
            dandiset = client.get_dandiset(DANDISET_ID, DANDISET_VERSION)
            candidate_paths = [a.path for a in dandiset.get_assets() if a.path.lower().endswith('.nwb')]
        print(f'Found {len(candidate_paths)} NWB assets in {DANDISET_ID} ({DANDISET_VERSION}).')
        for index, path in enumerate(candidate_paths[:MAX_ASSETS_TO_SHOW]):
            print(f'[{index}] {path}')
        if len(candidate_paths) > MAX_ASSETS_TO_SHOW:
            print(f'Only the first {MAX_ASSETS_TO_SHOW} paths are shown; refine in DANDI if necessary.')
        if SELECTED_ASSET_INDEX is not None:
            if isinstance(SELECTED_ASSET_INDEX, bool) or not isinstance(SELECTED_ASSET_INDEX, int):
                raise TypeError('SELECTED_ASSET_INDEX must be an integer printed above.')
            if not 0 <= SELECTED_ASSET_INDEX < len(candidate_paths):
                raise IndexError('SELECTED_ASSET_INDEX is outside the discovered NWB asset list.')
            selected_asset_path = candidate_paths[SELECTED_ASSET_INDEX]
            print('Explicitly selected:', selected_asset_path)
        else:
            print('Set SELECTED_ASSET_INDEX to one listed index; no remote file will be opened yet.')
    except Exception as error:
        print(f'Remote discovery failed; using local fallback: {error}')
else:
    print('No dandiset selected; using the local synthetic fallback.')

## Stream safely with `remfile`, `h5py`, and PyNWB

For each remote use, the helper resolves the explicitly selected path through DANDI again and obtains a fresh short-lived content URL. This follows the documented sequence: `remfile.File` for range reads, `h5py.File` for the HDF5 handle, then `NWBHDF5IO(file=h5_file)`. The temporary range cache and all handles close on exit. Metadata is read first; do not load an unknown acquisition wholesale.

In [ ]:
from contextlib import contextmanager
import tempfile
import h5py
from pynwb import NWBHDF5IO

def selected_asset_url(dandiset_id, dandiset_version, asset_path):
    from dandi.dandiapi import DandiAPIClient
    with DandiAPIClient() as client:
        dandiset = client.get_dandiset(dandiset_id, dandiset_version)
        asset = dandiset.get_asset_by_path(asset_path)
        return asset.get_content_url(follow_redirects=1, strip_query=True)

@contextmanager
def streamed_h5(dandiset_id, dandiset_version, asset_path):
    import remfile
    with tempfile.TemporaryDirectory(prefix='dandi-remfile-') as cache_dir:
        remote_file = remfile.File(
            selected_asset_url(dandiset_id, dandiset_version, asset_path),
            disk_cache=remfile.DiskCache(cache_dir),
        )
        try:
            with h5py.File(remote_file, 'r') as h5_file:
                yield h5_file
        finally:
            remote_file.close()

if selected_asset_path is not None:
    try:
        with streamed_h5(DANDISET_ID, DANDISET_VERSION, selected_asset_path) as h5_file:
            print({name: type(h5_file[name]).__name__ for name in h5_file.keys()})
    except Exception as error:
        print('Remote HDF5 inspection failed:', error)
else:
    print('Skipping remote inspection until a discovered path is explicitly selected.')

## Read compact NWB metadata, with a local fallback

NWB objects may lazily reference remote data, so use them while their I/O context is open. The default route writes a small synthetic file to a system temporary directory, reads its metadata, and removes it at cell completion—no large data or cache is committed to the repository.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
from pynwb import NWBFile
from pynwb.base import TimeSeries

@contextmanager
def synthetic_nwb_path():
    with tempfile.TemporaryDirectory() as directory:
        path = Path(directory) / 'synthetic_example.nwb'
        nwbfile = NWBFile('Synthetic streaming tutorial', 'SYNTHETIC_001', datetime.now(timezone.utc))
        nwbfile.add_acquisition(TimeSeries('voltage', data=np.sin(np.linspace(0, 8 * np.pi, 200)), unit='a.u.', rate=100.0))
        with NWBHDF5IO(str(path), 'w') as io:
            io.write(nwbfile)
        yield path

def metadata_summary(io):
    nwbfile = io.read()
    return {'identifier': nwbfile.identifier, 'session_description': nwbfile.session_description,
            'acquisitions': list(nwbfile.acquisition), 'processing_modules': list(nwbfile.processing)}

if selected_asset_path is not None:
    try:
        with streamed_h5(DANDISET_ID, DANDISET_VERSION, selected_asset_path) as h5_file:
            with NWBHDF5IO(file=h5_file, mode='r', load_namespaces=True) as io:
                print(metadata_summary(io))
    except Exception as error:
        print('Remote NWB read failed; run the local fallback cell below:', error)
else:
    with synthetic_nwb_path() as path:
        with NWBHDF5IO(str(path), 'r', load_namespaces=True) as io:
            print(metadata_summary(io))

## Request a bounded data slice

Inspect `list(nwbfile.acquisition)` for a selected remote asset, then set `ACQUISITION_NAME` only after confirming that the named object is a compatible `TimeSeries`. The code rejects an unset name and requests at most `MAX_SAMPLES`; the fallback always demonstrates the same bounded-slice discipline.

In [ ]:
ACQUISITION_NAME = None  # Set only after inspecting the selected NWB metadata.
MAX_SAMPLES = 100

if selected_asset_path is not None and ACQUISITION_NAME is not None:
    try:
        with streamed_h5(DANDISET_ID, DANDISET_VERSION, selected_asset_path) as h5_file:
            with NWBHDF5IO(file=h5_file, mode='r', load_namespaces=True) as io:
                nwbfile = io.read()
                if ACQUISITION_NAME not in nwbfile.acquisition:
                    raise KeyError(f'{ACQUISITION_NAME!r} is not an acquisition in this file.')
                series = nwbfile.acquisition[ACQUISITION_NAME]
                values = series.data[:MAX_SAMPLES]
                print('shape:', np.shape(values), 'unit:', getattr(series, 'unit', None))
    except Exception as error:
        print('Remote slice failed:', error)
else:
    with synthetic_nwb_path() as path:
        with NWBHDF5IO(str(path), 'r', load_namespaces=True) as io:
            series = io.read().acquisition['voltage']
            print('fallback first 10 samples:', np.round(series.data[:10], 3))

## References

- DANDI Archive: https://dandiarchive.org/
- DANDI Python API documentation: https://dandi.readthedocs.io/
- PyNWB streaming tutorial: https://pynwb.readthedocs.io/en/stable/tutorials/advanced_io/streaming.html
- Neurodata Without Borders documentation: https://nwb-schema.readthedocs.io/
- Teeters, J. L., et al. (2015). Neurodata Without Borders: Creating a Common Data Format for Neurophysiology. *Neuron*, 88(4), 629–634. https://doi.org/10.1016/j.neuron.2015.10.025
- Wagner, A. S., et al. (2022). The DANDI Archive: A neurophysiology data archive for publishing, sharing, and processing. *eLife*, 11, e78574. https://doi.org/10.7554/eLife.78574

## License

This notebook is licensed under the [Creative Commons Attribution 4.0 International License](https://creativecommons.org/licenses/by/4.0/). Each DANDI dandiset has its own license and citation requirements; review them before reuse or redistribution.